In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/tsovinarbabakhanyan/law-data/armenia_lawyer_routing_dataset_150.xlsx


In [2]:
import pandas as pd
from datasets import Dataset

df = pd.read_excel("/kaggle/input/datasets/tsovinarbabakhanyan/law-data/armenia_lawyer_routing_dataset_150.xlsx", sheet_name="train_data")

# Simple but effective prompt format for routing
def format_example(row):
    return {
        "text": f"""### Instruction:
Route the following legal query to the most appropriate lawyer type in Armenia.

Query: {row['text']}

### Response:
Recommended lawyer: {row['recommended_lawyer']}
Constitution articles: {row['constitution_articles']}
Routing hint: {row['routing_hint']}
""",
        "recommended_lawyer": row['recommended_lawyer']
    }

dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_example)
dataset = dataset.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [3]:
!pip install --no-deps "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers==0.0.28.post2
!pip install trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9n1p8rhu/unsloth_6be2e2b5e1484908827b149ee5f59c2d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9n1p8rhu/unsloth_6be2e2b5e1484908827b149ee5f59c2d
  Resolved https://github.com/unslothai/unsloth.git to commit a6c1f893fc87c0973f9c32e59ca3d7d54ffb9724
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.3.17-py3-none-any.whl size=29432256 sha256=9cec7f13d829c58f7876f7c8bd3c3f9e13bff11d5e5292028617ed1ccd7efc07
  Stored in directory: /tmp/pip-ephem-wheel-cache-e4szkl0b/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━

In [4]:
!pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: fsspec
    Found e

In [5]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

max_seq_length = 1024
dtype = None  # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# === FIXED TrainingArguments ===
training_args = TrainingArguments(
    per_device_train_batch_size=2,      # safer on T4
    gradient_accumulation_steps=4,      # effective batch size = 8
    warmup_steps=5,
    max_steps=120,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    
    # Key fixes for the 'int' .mean() error
    average_tokens_across_devices=False,
    dataloader_drop_last=False,
    save_strategy="no",
)

# Explicit data collator (helps stability with Gemma-2)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding="longest",
    return_tensors="pt"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"] if "test" in dataset else None,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,                    # Important: keep False for small 150-row dataset
    args=training_args,
    data_collator=data_collator,      # ← helps prevent loss type issues
)

# Train
trainer_stats = trainer.train()

print("Training completed successfully!")

# Save as GGUF (multiple quantizations)
model.save_pretrained_gguf(
    "armenia_lawyer_router_gguf",
    tokenizer,
    quantization_method=["q4_k_m", "q5_k_m", "q8_0"]
)

# Or just one common one:
# model.save_pretrained_gguf("armenia_lawyer_router", tokenizer, quantization_method="q4_k_m")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Gemma2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF f16 into q5_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.Q8_0.gguf', 'armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.Q5_K_M.gguf', 'armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.Q8_0.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to armenia_lawyer_router_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f armenia_lawyer_router_gguf_gguf/Modelfile


{'save_directory': 'armenia_lawyer_router_gguf',
 'gguf_directory': 'armenia_lawyer_router_gguf_gguf',
 'gguf_files': ['armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.Q8_0.gguf',
  'armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.Q5_K_M.gguf',
  'armenia_lawyer_router_gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf'],
 'modelfile_location': 'armenia_lawyer_router_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': True}

In [6]:
import os
print("GGUF files created:")
for f in os.listdir("armenia_lawyer_router_gguf"):
    if f.endswith(".gguf"):
        size = os.path.getsize(f"armenia_lawyer_router_gguf/{f}") / (1024**3)
        print(f"  {f}  →  {size:.2f} GB")

GGUF files created:


In [7]:
import os

folder = "armenia_lawyer_router_gguf"

print("✅ GGUF files created in folder:", folder)
print("-" * 60)

if os.path.exists(folder):
    files = [f for f in os.listdir(folder) if f.endswith(".gguf")]
    if files:
        for f in sorted(files):
            size_gb = os.path.getsize(os.path.join(folder, f)) / (1024 ** 3)
            print(f"• {f:50} → {size_gb:.2f} GB")
    else:
        print("No .gguf files found!")
else:
    print(f"Folder '{folder}' not found!")

✅ GGUF files created in folder: armenia_lawyer_router_gguf
------------------------------------------------------------
No .gguf files found!


In [8]:
import os
from IPython.display import FileLink

# 1. Zip the entire GGUF folder
!zip -r armenia_lawyer_router_gguf.zip armenia_lawyer_router_gguf

# 2. Show download link (click it!)
FileLink('armenia_lawyer_router_gguf.zip')

  adding: armenia_lawyer_router_gguf/ (stored 0%)
  adding: armenia_lawyer_router_gguf/config.json (deflated 71%)
  adding: armenia_lawyer_router_gguf/model.safetensors (deflated 21%)
  adding: armenia_lawyer_router_gguf/tokenizer_config.json (deflated 54%)
  adding: armenia_lawyer_router_gguf/tokenizer.json (deflated 84%)
  adding: armenia_lawyer_router_gguf/tokenizer.model (deflated 51%)
  adding: armenia_lawyer_router_gguf/.cache/ (stored 0%)
  adding: armenia_lawyer_router_gguf/.cache/huggingface/ (stored 0%)
  adding: armenia_lawyer_router_gguf/.cache/huggingface/.gitignore (stored 0%)
  adding: armenia_lawyer_router_gguf/.cache/huggingface/download/ (stored 0%)
  adding: armenia_lawyer_router_gguf/.cache/huggingface/download/tokenizer.model.metadata (deflated 30%)
  adding: armenia_lawyer_router_gguf/.cache/huggingface/download/model.safetensors.metadata (deflated 29%)
  adding: armenia_lawyer_router_gguf/chat_template.jinja (deflated 52%)


/kaggle/working/armenia_lawyer_router_gguf.zip